# Image encoding

## Preparing the dataset
Download the Chesapeake Bay Land Cover dataset and organize your dataset directory as recommended.

1. Copy `*_lc.tif` and `*_naip-new.tif` files for segmentation downstream tasks using s5cmd:
   ```bash
   # train
   s5cmd --no-sign-request cp --include "*_lc.tif" --include "*_naip-new.tif" "s3://us-west-2.opendata.source.coop/agentmorris/lila-wildlife/lcmcvpr2019/cvpr_chesapeake_landcover/ny_1m_2013_extended-debuffered-train_tiles/*" data/cvpr/files/train/

   # val
   s5cmd --no-sign-request cp --include "*_lc.tif" --include "*_naip-new.tif" "s3://us-west-2.opendata.source.coop/agentmorris/lila-wildlife/lcmcvpr2019/cvpr_chesapeake_landcover/ny_1m_2013_extended-debuffered-val_tiles/*" data/cvpr/files/val/

   # test
   s5cmd --no-sign-request cp --include "*_lc.tif" --include "*_naip-new.tif" "s3://us-west-2.opendata.source.coop/agentmorris/lila-wildlife/lcmcvpr2019/cvpr_chesapeake_landcover/ny_1m_2013_extended-debuffered-test_tiles/*" data/cvpr/files/test/
   ```

2. Create chips of size `224 x 224` to feed them to the model:
    ```bash
    python claymodel/finetune/segment/preprocess_data.py data/cvpr/files data/cvpr/ny 224
    ```

Directory structure:
```
data/
└── cvpr/
    ├── files/
    │   ├── train/
    │   ├── val/
    │   └── test/
    └── ny/
        ├── train/
        │   ├── chips/
        │   └── labels/
        ├── val/
        │   ├── chips/
        │   └── labels/
        └── test/
            ├── chips/
            └── labels/
```

## Using the model

1. Download the Clay model checkpoint from [Huggingface model hub](https://huggingface.co/made-with-clay/Clay/blob/main/v1.5/clay-v1.5.ckpt) and save it in the `checkpoints/` directory.

2. Run the model. Configuration file can be found in `configs/extract_embeddings_chesapeake.yaml`. Beaware, the function Encode_images_from_config needs to be run for all three folders "train", "val" and "test". For this you will need to modify the fields "predict_chip_dir", "predict_label_dir" and "output_dir" from the configuration file.

In [1]:
import os
import shutil
from pathlib import Path

def find_project_root(marker='claymodel'):
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Project root not found")

os.chdir(find_project_root())

model_dir = "checkpoints/"
checkpoint_filename = "clay-v1.5.ckpt"
full_target_path = os.path.join(model_dir, checkpoint_filename)

if os.path.exists(full_target_path):
    print(f"✓ File already exists at: {full_target_path}")
else:
    print(f"✗ File not found at: {full_target_path}")
    print("Downloading checkpoint...")
    
    # Create target directory if it doesn't exist
    os.makedirs(model_dir, exist_ok=True)
    
    # Download the file
    !wget -q https://huggingface.co/made-with-clay/Clay/resolve/main/v1.5/clay-v1.5.ckpt
    
    # Move the downloaded file to the target location
    if os.path.exists(checkpoint_filename):
        shutil.move(checkpoint_filename, full_target_path)
        print(f"✓ Successfully downloaded and moved to: {full_target_path}")
    else:
        print("✗ Download failed - file not found after wget")


✓ File already exists at: checkpoints/clay-v1.5.ckpt


In [2]:
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.cli import LightningArgumentParser, instantiate_class

from claymodel.finetune.segment.embedding_extractor import ClayEmbeddingExtractor
from claymodel.finetune.segment.chesapeake_datamodule_v2 import ChesapeakeDataModule

seed_everything(42)  # your seed here

def Encode_images_from_config(config_path, dataset="train"):
    
    parser = LightningArgumentParser()
    parser.add_lightning_class_args(ClayEmbeddingExtractor, "model")
    parser.add_lightning_class_args(ChesapeakeDataModule, "data")
    parser.add_lightning_class_args(Trainer, "trainer")
    
    config = parser.parse_path(config_path)
    trainer_config = dict(config["trainer"])
    
    callbacks = []
    for callback_config in trainer_config["callbacks"]:
        if hasattr(callback_config, 'class_path') and hasattr(callback_config, 'init_args'):
            # This is a Namespace object from CLI parsing
            callback = instantiate_class((), callback_config)
            callbacks.append(callback)
        elif isinstance(callback_config, dict) and "class_path" in callback_config:
            # This is a dictionary configuration
            callback = instantiate_class((), callback_config)
            callbacks.append(callback)
        else:
            # Already an instantiated callback
            callbacks.append(callback_config)
    
    trainer_config["callbacks"] = callbacks        
    
    # Create instances
    model = ClayEmbeddingExtractor(**config["model"])
    config["data"]["predict_ds"] = dataset
    datamodule = ChesapeakeDataModule(**config["data"])
    trainer = Trainer(**trainer_config)

    trainer.predict(model, datamodule)   

    return config

Seed set to 42


In [ ]:
available_datasets = ["train", "val", "test"]
for ds in available_datasets:
    print(f"Encoding dataset: {ds}")
    Encode_images_from_config('configs/extract_embeddings_chesapeake.yaml', dataset=ds)

Encoding dataset: train
🗜️  Saving in float16


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


✅ Clay model loaded from: checkpoints/clay-v1.5.ckpt
🔒 Encoder frozen: True


/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


🚀 Starting embedding extraction...


Predicting: |          | 0/? [00:00<?, ?it/s]

💾 Saving embeddings for batch 0 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 1 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 2 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 3 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 4 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 5 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 6 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 7 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 8 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 9 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 10 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 11 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 12 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 13 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 14 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 15 (shape: (32, 784, 1024))
💾 Saving embeddings for batch 16 (shape: (32, 784, 1024))
💾 Saving embeddings for 